In [ ]:
import pandas as pd
import numpy as np
import cpi
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load Data

In [ ]:
artnet_2024 = ** Tabular Data**

In [ ]:
def convert_price(i,row):
    price = cpi.inflate(row['sale price usd'], row['sale year'], to=2024)
    return i, price

In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
prices = np.zeros(artnet_2024.shape[0], dtype=object)
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_price, i, artnet_2024.iloc[i]): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, price = fut.result()
            prices[i] =price
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")

    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
artnet_2024['price'] = prices

In [ ]:
# number_size = artnet_2024.shape[0]
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
prices = np.zeros(N, dtype=float)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_price, i, artnet_2024.iloc[i]): i
        for i in range(range_start,range_end)
    }
    completed = 0
    for fut in as_completed(futures):
        i, price = fut.result()
        prices[i-range_start]=price
        completed +=1
        if completed % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
artnet_2024['price'] = prices

In [ ]:
artnet_2024.columns

# Merge same artwork with values

In [ ]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe2024_price.xlsx")

In [ ]:
artnet_2024.columns

In [ ]:
artnet_2024_merged = artnet_2024.groupby("artwork id").agg({
    "artwork id":"first",
    "artist id": "first",
    "first": "first",
    "last": "first",
    "nationality":'first',
    "year born":"first",
    "year died":"first",
    "title": "first",
    "workyear modifier":"first",
    "workyear from":"first",
    "workyear to":"first",
    "est lo usd":"max",
    "est hi usd":"max",
    "sale price usd":"max",
    "artist":"first",
    "price":"max"})